# LeetCode 3162 — Find the Number of Good Pairs I bằng Python

Link đề: https://leetcode.com/problems/find-the-number-of-good-pairs-i/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, phép chia hết

Skeleton LeetCode:

```python
class Solution:
    def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một cặp chỉ số `(i, j)` là **tốt (good)** khi `nums1[i]` **chia hết** cho `nums2[j] * k`; ta phải **đếm tổng số cặp tốt** (đếm theo **chỉ số**, nên giá trị trùng nhau vẫn được tính nhiều lần).
2. Nắm các công cụ Python: toán tử **`%`** để kiểm tra chia hết, **hai vòng `for` lồng nhau**, `sum(... for ...)` trên biểu thức đúng/sai (`True = 1`), `itertools.product`, `collections.Counter`, và `range(a, b, bước)` để liệt kê **bội số**.
3. Đi từ cách **hai vòng lặp thuần** → **gom bằng `sum`/generator** → **tính trước `nums2[j] * k` và cắt tỉa** → **đếm theo giá trị khác nhau bằng `Counter`** → **liệt kê bội số** của `nums2[j] * k`.
4. Hiểu mấu chốt: ràng buộc nhỏ (`n, m <= 50`) cho phép vét cạn `n * m` cặp, nhưng khi có **nhiều giá trị trùng**, đếm theo **(giá trị, số lần)** gọn hơn và là bước đệm cho bài **Good Pairs II (3164)** với dữ liệu lớn.
5. Tránh các lỗi kinh điển: viết sai **thứ tự ưu tiên** `nums1[i] % nums2[j] * k` (thực chất là `(x % y) * k`), **quên nhân `k`**, đảo vai trò `nums1`/`nums2`, và dùng `/` (chia số thực) thay vì `%`.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2` có độ dài lần lượt là `n` và `m`, và một số nguyên **dương** `k`.

Một cặp `(i, j)` được gọi là **tốt** nếu `nums1[i]` **chia hết** cho `nums2[j] * k` (với `0 <= i <= n - 1`, `0 <= j <= m - 1`).

Trả về **tổng số cặp tốt**.

**Ví dụ 1**

- Input: `nums1 = [1,3,4]`, `nums2 = [1,3,4]`, `k = 1`
- Output: `5`
- Giải thích: 5 cặp tốt là `(0, 0)`, `(1, 0)`, `(1, 1)`, `(2, 0)` và `(2, 2)`.

**Ví dụ 2**

- Input: `nums1 = [1,2,4,12]`, `nums2 = [2,4]`, `k = 3`
- Output: `2`
- Giải thích: 2 cặp tốt là `(3, 0)` và `(3, 1)` (vì `12` chia hết cho `2 * 3 = 6` và cho `4 * 3 = 12`).

**Ràng buộc:** `1 <= n, m <= 50`, `1 <= nums1[i], nums2[j] <= 50`, `1 <= k <= 50`.

In [1]:
from typing import List
from collections import Counter
from itertools import product
import random

examples = [
    ([1, 3, 4], [1, 3, 4], 1, 5),
    ([1, 2, 4, 12], [2, 4], 3, 2),
]
for nums1, nums2, k, expected in examples:
    print(nums1, nums2, "k =", k, "-> kỳ vọng", expected)

[1, 3, 4] [1, 3, 4] k = 1 -> kỳ vọng 5
[1, 2, 4, 12] [2, 4] k = 3 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. "Chia hết" và toán tử `%`

`a` **chia hết** cho `b` khi phần dư `a % b` bằng `0`. Ví dụ `12 % 4 = 0` (chia hết), `12 % 5 = 2` (không chia hết). Lưu ý `%` không cho phép `b = 0`; ở bài này `b = nums2[j] * k >= 1` nên an toàn.

In [2]:
for a, b in [(12, 4), (12, 5), (12, 6), (12, 12), (4, 12), (7, 1)]:
    print(f"{a} % {b} = {a % b} -> {'chia hết' if a % b == 0 else 'không chia hết'}")

12 % 4 = 0 -> chia hết
12 % 5 = 2 -> không chia hết
12 % 6 = 0 -> chia hết
12 % 12 = 0 -> chia hết
4 % 12 = 4 -> không chia hết
7 % 1 = 0 -> chia hết


### 3.2. Thứ tự ưu tiên: `x % (y * k)` khác `x % y * k`

Trong Python `%` và `*` có **cùng mức ưu tiên** và được tính **từ trái sang phải**. Vì vậy `x % y * k` nghĩa là `(x % y) * k` — **không** phải `x % (y * k)`. Phải đặt **ngoặc**: `x % (y * k) == 0`.

In [3]:
x, y, k = 12, 2, 3
print(x % y * k)        # (12 % 2) * 3 = 0  -> trông như "chia hết" nhưng không đúng đề
print(x % (y * k))      # 12 % 6 = 0
x, y, k = 4, 2, 3
print(x % y * k, "(sai: ra 0, tưởng chia hết)")     # (4 % 2) * 3 = 0
print(x % (y * k), "(đúng: 4 % 6 = 4, không chia hết)")

0
0
0 (sai: ra 0, tưởng chia hết)
4 (đúng: 4 % 6 = 4, không chia hết)


### 3.3. Hai vòng `for` lồng nhau để duyệt mọi cặp

Muốn xét **mọi cặp** `(i, j)` ta cho vòng ngoài chạy trên `nums1` và vòng trong chạy trên `nums2`. Có `n * m` cặp. `itertools.product(a, b)` làm đúng việc đó trong một dòng.

In [4]:
nums1, nums2 = [1, 2], [5, 6, 7]
for i, x in enumerate(nums1):
    for j, y in enumerate(nums2):
        print(f"(i={i}, j={j}) -> ({x}, {y})")
print(len(list(product(nums1, nums2))), "cặp =", len(nums1), "*", len(nums2))

(i=0, j=0) -> (1, 5)
(i=0, j=1) -> (1, 6)
(i=0, j=2) -> (1, 7)
(i=1, j=0) -> (2, 5)
(i=1, j=1) -> (2, 6)
(i=1, j=2) -> (2, 7)
6 cặp = 2 * 3


### 3.4. Đếm bằng `sum` trên biểu thức đúng/sai

Trong Python `True` bằng `1` và `False` bằng `0`, nên `sum(điều_kiện for ...)` **đếm số lần điều kiện đúng**. Ví dụ `sum(x % 2 == 0 for x in [1, 2, 4])` bằng `2`.

In [5]:
print(True + True + False)                        # 2
print(sum(x % 2 == 0 for x in [1, 2, 4, 7, 10]))  # 3 số chẵn

2
3


### 3.5. `Counter`: đếm theo giá trị

Nếu nhiều phần tử có cùng giá trị thì mọi phép kiểm tra cho chúng **giống hệt nhau**. `Counter(nums)` cho `{giá trị: số lần}`; ta chỉ kiểm tra **mỗi giá trị khác nhau một lần**, rồi **nhân với số lần** của nó.

In [6]:
c = Counter([2, 2, 5, 2, 5])
print(c)                          # Counter({2: 3, 5: 2})
print(c[2], c[5], c[99])          # 3 2 0 : giá trị vắng mặt cho 0 (và không bị thêm vào Counter)
print(len(c))                     # 2 giá trị khác nhau

Counter({2: 3, 5: 2})
3 2 0
2


### 3.6. Bội số của `d` bằng `range(d, limit + 1, d)`

Các **bội số** của `d` không vượt quá `limit` là `d, 2d, 3d, ...`: `range(d, limit + 1, d)`. Nhớ `limit + 1` vì cận trên của `range` bị loại. Đây là cách "đi ngược": thay vì hỏi "`x` có chia hết cho `d` không?", ta liệt kê **mọi `x` chia hết cho `d`**.

In [7]:
print(list(range(6, 50 + 1, 6)))      # [6, 12, 18, 24, 30, 36, 42, 48]
print(list(range(12, 12 + 1, 12)))    # [12] : đúng bằng limit vẫn được lấy
print(list(range(20, 12 + 1, 20)))    # []   : bội số nhỏ nhất đã vượt limit

[6, 12, 18, 24, 30, 36, 42, 48]
[12]
[]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với Ví dụ 2: `nums1 = [1, 2, 4, 12]`, `nums2 = [2, 4]`, `k = 3`. Trước hết tính **số chia** `d = nums2[j] * k`: `2 * 3 = 6` và `4 * 3 = 12`. Sau đó kiểm tra từng `nums1[i]` với từng `d`:

| `nums1[i]` | chia hết cho 6? | chia hết cho 12? | số cặp tốt |
|---|---|---|---|
| 1 | không | không | 0 |
| 2 | không | không | 0 |
| 4 | không | không | 0 |
| 12 | có | có | 2 |

Tổng `0 + 0 + 0 + 2 = 2`.

Nhận xét: `d = nums2[j] * k` chỉ phụ thuộc `j`, ta có thể tính **một lần** cho mỗi `j` rồi dùng lại cho mọi `i`.

In [8]:
nums1, nums2, k = [1, 2, 4, 12], [2, 4], 3
divisors = [y * k for y in nums2]
print("các số chia d = y * k:", divisors)
for x in nums1:
    flags = [x % d == 0 for d in divisors]
    print(f"x = {x:>2}: chia hết cho {divisors}? {flags} -> {sum(flags)} cặp tốt")

các số chia d = y * k: [6, 12]
x =  1: chia hết cho [6, 12]? [False, False] -> 0 cặp tốt
x =  2: chia hết cho [6, 12]? [False, False] -> 0 cặp tốt
x =  4: chia hết cho [6, 12]? [False, False] -> 0 cặp tốt
x = 12: chia hết cho [6, 12]? [True, True] -> 2 cặp tốt


## 5. Cách 1 — Brute Force / baseline: hai vòng lặp và đếm

**Ý tưởng.** Bám sát đề: với mọi `i`, mọi `j`, nếu `nums1[i] % (nums2[j] * k) == 0` thì tăng biến đếm. Đây là cách trực tiếp nhất và với `n, m <= 50` chỉ có tối đa `2500` cặp.

In [9]:
def number_of_pairs_loops(nums1: List[int], nums2: List[int], k: int) -> int:
    count = 0
    for i in range(len(nums1)):
        for j in range(len(nums2)):
            if nums1[i] % (nums2[j] * k) == 0:        # nhớ ngoặc quanh nums2[j] * k
                count += 1
    return count


for nums1, nums2, k, expected in examples:
    print(number_of_pairs_loops(nums1, nums2, k), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
2 (kỳ vọng 2 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums1)`, `m = len(nums2)`. Hai vòng lồng nhau, mỗi cặp một phép `%` ⇒ **Time `O(n * m)`**, **Space `O(1)`**. Với giới hạn đề là đủ nhanh.

**Nút thắt:** `nums2[j] * k` bị tính lại `n` lần cho mỗi `j`; và nếu có nhiều giá trị trùng nhau thì ta kiểm tra lặp lại những phép giống hệt nhau.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai cải tiến độc lập:

1. **Tính trước số chia** `d_j = nums2[j] * k` một lần, rồi mỗi `x` chỉ cần `x % d == 0`. Thêm một **cắt tỉa**: vì `x >= 1`, nếu `d > x` thì `x` không thể chia hết cho `d` (chia hết đòi hỏi `d <= x`), khỏi cần tính `%`.
2. **Đếm theo giá trị khác nhau:** nếu `x` xuất hiện `c1` lần trong `nums1` và `y` xuất hiện `c2` lần trong `nums2` và `x % (y * k) == 0`, thì cặp giá trị `(x, y)` đóng góp **`c1 * c2` cặp chỉ số tốt**. Dùng `Counter` ⇒ số phép kiểm tra giảm xuống `(số giá trị khác nhau của nums1) * (số giá trị khác nhau của nums2)`.

Ngoài ra còn cách "đi ngược": với mỗi giá trị `y` của `nums2`, liệt kê **mọi bội số của `y * k`** (không vượt quá `max(nums1)`) và cộng số lần xuất hiện của chúng trong `nums1`.

In [10]:
nums1, nums2, k = [1, 3, 4], [1, 3, 4], 1
c1, c2 = Counter(nums1), Counter(nums2)
total = 0
for x, a in c1.items():
    for y, b in c2.items():
        if x % (y * k) == 0:
            print(f"giá trị ({x}, {y}): {a} * {b} = {a * b} cặp chỉ số tốt")
            total += a * b
print("tổng:", total)

giá trị (1, 1): 1 * 1 = 1 cặp chỉ số tốt
giá trị (3, 1): 1 * 1 = 1 cặp chỉ số tốt
giá trị (3, 3): 1 * 1 = 1 cặp chỉ số tốt
giá trị (4, 1): 1 * 1 = 1 cặp chỉ số tốt
giá trị (4, 4): 1 * 1 = 1 cặp chỉ số tốt
tổng: 5


## 7. Các cách tiếp theo

### Cách 2 — Gom bằng `sum` + generator

**Giải thích từng đoạn.**

1. `for x in nums1 for y in nums2`: duyệt mọi cặp giá trị (không cần chỉ số).
2. `x % (y * k) == 0`: biểu thức đúng/sai.
3. `sum(...)`: cộng `True`/`False` ⇒ số cặp tốt.

In [11]:
def number_of_pairs_sum(nums1: List[int], nums2: List[int], k: int) -> int:
    return sum(x % (y * k) == 0 for x in nums1 for y in nums2)


for nums1, nums2, k, expected in examples:
    print(number_of_pairs_sum(nums1, nums2, k), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
2 (kỳ vọng 2 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionSum:
    def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int:
        return sum(x % (y * k) == 0 for x in nums1 for y in nums2)


print(SolutionSum().numberOfPairs([1, 2, 4, 12], [2, 4], 3))      # 2

2


**Độ phức tạp Cách 2:** **Time `O(n * m)`**, **Space `O(1)`** (generator không tạo danh sách).

### Cách 3 — Tính trước `d = nums2[j] * k` và cắt tỉa `d > x`

**Giải thích.** Tạo `divisors = [y * k for y in nums2]` một lần. Với mỗi `x`, chỉ xét `d <= x` (nếu `d > x` thì chắc chắn không chia hết vì `x >= 1`). Cách này giảm số phép `%`, nhưng độ phức tạp tiệm cận vẫn là `O(n * m)`.

In [13]:
def number_of_pairs_precompute(nums1: List[int], nums2: List[int], k: int) -> int:
    divisors = [y * k for y in nums2]          # d_j = nums2[j] * k, tính một lần
    count = 0
    for x in nums1:
        for d in divisors:
            if d <= x and x % d == 0:          # d > x thì x không thể chia hết cho d
                count += 1
    return count


for nums1, nums2, k, expected in examples:
    print(number_of_pairs_precompute(nums1, nums2, k), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
2 (kỳ vọng 2 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionPrecompute:
    def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int:
        divisors = [y * k for y in nums2]
        count = 0
        for x in nums1:
            for d in divisors:
                if d <= x and x % d == 0:
                    count += 1
        return count


print(SolutionPrecompute().numberOfPairs([1, 3, 4], [1, 3, 4], 1))     # 5

5


**Độ phức tạp Cách 3:** **Time `O(n * m)`** (trường hợp xấu nhất), **Space `O(m)`** (danh sách `divisors`).

### Cách 4 — Đếm theo giá trị khác nhau bằng `Counter`

**Giải thích.** `c1 = Counter(nums1)`, `c2 = Counter(nums2)`. Với mỗi giá trị khác nhau `x` (xuất hiện `a` lần) và `y` (xuất hiện `b` lần), nếu `x % (y * k) == 0` thì cộng `a * b`. Khi có nhiều số trùng, ta kiểm tra ít cặp hơn. Đây là tiền đề cho bài II với `n, m` lớn.

In [15]:
def number_of_pairs_counter(nums1: List[int], nums2: List[int], k: int) -> int:
    c1, c2 = Counter(nums1), Counter(nums2)
    total = 0
    for x, a in c1.items():
        for y, b in c2.items():
            if x % (y * k) == 0:
                total += a * b              # a cách chọn i, b cách chọn j
    return total


for nums1, nums2, k, expected in examples:
    print(number_of_pairs_counter(nums1, nums2, k), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
2 (kỳ vọng 2 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionCounter:
    def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int:
        c1, c2 = Counter(nums1), Counter(nums2)
        total = 0
        for x, a in c1.items():
            for y, b in c2.items():
                if x % (y * k) == 0:
                    total += a * b
        return total


print(SolutionCounter().numberOfPairs([1, 2, 4, 12], [2, 4], 3))     # 2

2


**Độ phức tạp Cách 4:** **Time `O(u1 * u2)`** với `u1`, `u2` là số giá trị khác nhau (`<= n`, `<= m`, cũng `<= 50`), **Space `O(u1 + u2)`**.

### Cách 5 — Liệt kê bội số của `y * k`

**Giải thích.** Đi ngược lại: với mỗi giá trị khác nhau `y` của `nums2` (xuất hiện `b` lần), đặt `d = y * k`; các `x` thỏa điều kiện chính là **bội số của `d`**: `d, 2d, 3d, ...` tới `max(nums1)`. Với mỗi bội số `m`, cộng `c1[m] * b`. Cách này nhanh khi `d` lớn (ít bội số), và là kỹ thuật chuẩn cho dữ liệu lớn (tổng số bước theo chuỗi điều hòa `limit/1 + limit/2 + ...`).

In [17]:
def number_of_pairs_multiples(nums1: List[int], nums2: List[int], k: int) -> int:
    c1, c2 = Counter(nums1), Counter(nums2)
    limit = max(nums1)
    total = 0
    for y, b in c2.items():
        d = y * k
        for multiple in range(d, limit + 1, d):      # d, 2d, 3d, ... <= max(nums1)
            total += c1[multiple] * b                # c1[multiple] = 0 nếu nums1 không có giá trị đó
    return total


for nums1, nums2, k, expected in examples:
    print(number_of_pairs_multiples(nums1, nums2, k), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
2 (kỳ vọng 2 )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionMultiples:
    def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int:
        c1, c2 = Counter(nums1), Counter(nums2)
        limit = max(nums1)
        total = 0
        for y, b in c2.items():
            d = y * k
            for multiple in range(d, limit + 1, d):
                total += c1[multiple] * b
        return total


print(SolutionMultiples().numberOfPairs([1, 3, 4], [1, 3, 4], 1))     # 5

5


**Độ phức tạp Cách 5:** **Time `O(u2 * limit / (k * min(nums2)) + n + m)`** — với mỗi `y` ta đi `limit / (y * k)` bước; trường hợp xấu nhất khi `k = 1` và `y = 1` là `O(limit)` mỗi `y`, tổng `O(u2 * limit)` nhưng thực tế nhỏ hơn nhờ chuỗi điều hòa. **Space `O(u1 + u2)`**.

## 8. TRACE từng bước

Ta trace Cách 1 (hai vòng lặp) trên Ví dụ 2 và Cách 5 (bội số) trên Ví dụ 1.

In [19]:
def trace_loops(nums1: List[int], nums2: List[int], k: int) -> int:
    print(f"nums1 = {nums1}, nums2 = {nums2}, k = {k}")
    count = 0
    for i, x in enumerate(nums1):
        for j, y in enumerate(nums2):
            d = y * k
            good = x % d == 0
            if good:
                count += 1
            print(f"  (i={i}, j={j}): x={x}, y={y}, d=y*k={d}, x % d = {x % d} -> {'TỐT' if good else 'không'}, count = {count}")
    return count


assert trace_loops([1, 2, 4, 12], [2, 4], 3) == 2

nums1 = [1, 2, 4, 12], nums2 = [2, 4], k = 3
  (i=0, j=0): x=1, y=2, d=y*k=6, x % d = 1 -> không, count = 0
  (i=0, j=1): x=1, y=4, d=y*k=12, x % d = 1 -> không, count = 0
  (i=1, j=0): x=2, y=2, d=y*k=6, x % d = 2 -> không, count = 0
  (i=1, j=1): x=2, y=4, d=y*k=12, x % d = 2 -> không, count = 0
  (i=2, j=0): x=4, y=2, d=y*k=6, x % d = 4 -> không, count = 0
  (i=2, j=1): x=4, y=4, d=y*k=12, x % d = 4 -> không, count = 0
  (i=3, j=0): x=12, y=2, d=y*k=6, x % d = 0 -> TỐT, count = 1
  (i=3, j=1): x=12, y=4, d=y*k=12, x % d = 0 -> TỐT, count = 2


In [20]:
def trace_multiples(nums1: List[int], nums2: List[int], k: int) -> int:
    c1, c2 = Counter(nums1), Counter(nums2)
    limit = max(nums1)
    print(f"c1 = {dict(c1)}, c2 = {dict(c2)}, limit = max(nums1) = {limit}")
    total = 0
    for y, b in c2.items():
        d = y * k
        multiples = list(range(d, limit + 1, d))
        print(f"  y={y} (xuất hiện {b} lần): d={d}, các bội số <= {limit}: {multiples}")
        for m in multiples:
            add = c1[m] * b
            total += add
            print(f"    bội số {m}: c1[{m}]={c1[m]}, cộng {c1[m]} * {b} = {add}, tổng = {total}")
    return total


assert trace_multiples([1, 3, 4], [1, 3, 4], 1) == 5

c1 = {1: 1, 3: 1, 4: 1}, c2 = {1: 1, 3: 1, 4: 1}, limit = max(nums1) = 4
  y=1 (xuất hiện 1 lần): d=1, các bội số <= 4: [1, 2, 3, 4]
    bội số 1: c1[1]=1, cộng 1 * 1 = 1, tổng = 1
    bội số 2: c1[2]=0, cộng 0 * 1 = 0, tổng = 1
    bội số 3: c1[3]=1, cộng 1 * 1 = 1, tổng = 2
    bội số 4: c1[4]=1, cộng 1 * 1 = 1, tổng = 3
  y=3 (xuất hiện 1 lần): d=3, các bội số <= 4: [3]
    bội số 3: c1[3]=1, cộng 1 * 1 = 1, tổng = 4
  y=4 (xuất hiện 1 lần): d=4, các bội số <= 4: [4]
    bội số 4: c1[4]=1, cộng 1 * 1 = 1, tổng = 5


Bảng trace Cách 1 với `nums1 = [1, 2, 4, 12]`, `nums2 = [2, 4]`, `k = 3` (`d = y * k` là `6` cho `y = 2` và `12` cho `y = 4`):

| `i` | `x` | `d = 6` (y=2) | `d = 12` (y=4) | cặp tốt thêm |
|---|---|---|---|---|
| 0 | 1 | `1 % 6 = 1` | `1 % 12 = 1` | 0 |
| 1 | 2 | `2 % 6 = 2` | `2 % 12 = 2` | 0 |
| 2 | 4 | `4 % 6 = 4` | `4 % 12 = 4` | 0 |
| 3 | 12 | `12 % 6 = 0` | `12 % 12 = 0` | **2** |

Đáp án `2`, đúng hai cặp `(3, 0)` và `(3, 1)`.

Cách 5 với Ví dụ 1 (`k = 1`): `y = 1` ⇒ bội số `1, 2, 3, 4` ⇒ `c1[1] + c1[3] + c1[4] = 1 + 1 + 1 = 3`; `y = 3` ⇒ bội số `3` ⇒ `1`; `y = 4` ⇒ bội số `4` ⇒ `1`. Tổng `3 + 1 + 1 = 5`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 1 viết gọn (hai vòng lặp trên giá trị + `%`): đơn giản, đúng, `O(n * m)` — thừa nhanh với `n, m <= 50`.

In [21]:
class Solution:
    def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int:
        count = 0
        for x in nums1:                          # chọn i
            for y in nums2:                      # chọn j
                if x % (y * k) == 0:             # x chia hết cho y * k
                    count += 1
        return count


sol = Solution()
for nums1, nums2, k, expected in examples:
    print(sol.numberOfPairs(nums1, nums2, k), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
2 (kỳ vọng 2 )


## 10. Vì sao thuật toán đúng

- **Đếm theo chỉ số:** đề đếm các cặp `(i, j)`. Hai vòng lặp duyệt **mỗi** cặp chỉ số **đúng một lần**, nên không bỏ sót và không đếm trùng — kể cả khi các giá trị lặp (ví dụ `nums1 = [2, 2]` có hai chỉ số khác nhau nên đóng góp hai cặp).
- **Điều kiện đúng đề:** `nums1[i]` chia hết cho `nums2[j] * k` ⇔ `nums1[i] % (nums2[j] * k) == 0`; mẫu số luôn `>= 1` nên không bao giờ chia cho `0`.
- **Cách 4 (Counter):** các chỉ số có cùng giá trị cho cùng kết quả kiểm tra; có `a` chỉ số `i` ứng với `x` và `b` chỉ số `j` ứng với `y` nên có `a * b` cặp chỉ số — đúng bằng đóng góp `a * b`.
- **Cách 5 (bội số):** `x` chia hết cho `d = y * k` ⇔ `x` là một bội số dương của `d`; `x <= max(nums1)` nên chỉ cần liệt kê bội số tới `max(nums1)`, và `c1[m] = 0` cho bội số không có trong `nums1`.
- **Cắt tỉa ở Cách 3:** `x >= 1`, nên nếu `d > x` thì `0 < x < d` và `x` không chia hết cho `d`; bỏ qua không đổi kết quả.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums1)`, `m = len(nums2)`, `u1`, `u2` là số giá trị khác nhau (`u1 <= n`, `u2 <= m`), `V = 50` là giá trị lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Hai vòng `for` | `O(n * m)` | `O(1)` |
| 2. `sum` + generator | `O(n * m)` | `O(1)` |
| 3. Tính trước `d`, cắt tỉa | `O(n * m)` | `O(m)` |
| 4. `Counter` hai bên | `O(u1 * u2)` | `O(u1 + u2)` |
| 5. Liệt kê bội số | `~O(u2 * V / (k * y))` mỗi `y`, thực tế nhỏ | `O(u1 + u2)` |

Tối đa `50 * 50 = 2500` phép kiểm tra: mọi cách đều tức thì.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [22]:
sol = Solution()
impls = {
    "loops": number_of_pairs_loops,
    "sum": number_of_pairs_sum,
    "precompute": number_of_pairs_precompute,
    "counter": number_of_pairs_counter,
    "multiples": number_of_pairs_multiples,
    "leetcode": sol.numberOfPairs,
}


def check_all(nums1, nums2, k, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums1), list(nums2), k)
        assert actual == expected, (label, key, nums1, nums2, k, actual, expected)
    print("PASS", label)


for idx, (nums1, nums2, k, expected) in enumerate(examples, 1):
    check_all(nums1, nums2, k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [23]:
check_all([1], [1], 1, 1, "một phần tử mỗi mảng, 1 chia hết cho 1")
check_all([1], [1], 2, 0, "1 không chia hết cho 1 * 2")
check_all([50], [50], 1, 1, "giá trị lớn nhất, k = 1")
check_all([50], [50], 2, 0, "50 không chia hết cho 100")
check_all([50], [1], 50, 1, "k = 50, 50 chia hết cho 1 * 50")
check_all([2, 2, 2], [1], 1, 3, "giá trị trùng: tính theo chỉ số")
check_all([6], [1, 1, 1], 1, 3, "nums2 có giá trị trùng")
check_all([2, 2], [1, 1], 1, 4, "trùng cả hai bên: 2 * 2 = 4 cặp")
check_all([12], [1, 2, 3, 4, 6, 12], 1, 6, "mọi ước của 12")
check_all([12], [1, 2, 3, 4, 6, 12], 2, 4, "ước của 12 sau khi nhân 2 (d = 2, 4, 6, 8, 12, 24)")
check_all([1, 2, 3, 4, 5], [7], 1, 0, "mọi x đều nhỏ hơn d")
check_all(list(range(1, 51)), [1], 1, 50, "mọi số chia hết cho 1")
check_all(list(range(1, 51)), [2], 1, 25, "25 số chẵn")

PASS một phần tử mỗi mảng, 1 chia hết cho 1
PASS 1 không chia hết cho 1 * 2
PASS giá trị lớn nhất, k = 1
PASS 50 không chia hết cho 100
PASS k = 50, 50 chia hết cho 1 * 50
PASS giá trị trùng: tính theo chỉ số
PASS nums2 có giá trị trùng
PASS trùng cả hai bên: 2 * 2 = 4 cặp
PASS mọi ước của 12
PASS ước của 12 sau khi nhân 2 (d = 2, 4, 6, 8, 12, 24)
PASS mọi x đều nhỏ hơn d
PASS mọi số chia hết cho 1
PASS 25 số chẵn


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **thiếu ngoặc** `x % y * k == 0`; (b) **quên nhân `k`** (`x % y == 0`).

In [24]:
def wrong_precedence(nums1, nums2, k):
    return sum(x % y * k == 0 for x in nums1 for y in nums2)          # SAI: (x % y) * k


def wrong_forget_k(nums1, nums2, k):
    return sum(x % y == 0 for x in nums1 for y in nums2)              # SAI: quên k


print(wrong_precedence([1, 2, 4, 12], [2, 4], 3), "!=", number_of_pairs_loops([1, 2, 4, 12], [2, 4], 3))
print(wrong_forget_k([1, 2, 4, 12], [2, 4], 3), "!=", number_of_pairs_loops([1, 2, 4, 12], [2, 4], 3))
assert wrong_precedence([1, 2, 4, 12], [2, 4], 3) != number_of_pairs_loops([1, 2, 4, 12], [2, 4], 3)
assert wrong_forget_k([1, 2, 4, 12], [2, 4], 3) != number_of_pairs_loops([1, 2, 4, 12], [2, 4], 3)

5 != 2
5 != 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: `n, m` trong `1..50`, giá trị `1..50`, `k` trong `1..50`. Vì `k` lớn làm đáp án thường bằng `0`, ta chọn `k` thiên về giá trị nhỏ và còn thử các tập giá trị "nhiều ước" (như `1, 2, 3, 4, 6, 12, 24, 48`) để đáp án khác `0`. Đối chiếu với oracle `itertools.product`. Ngoài ra **vét cạn** mọi `nums1`, `nums2` nhỏ.

In [25]:
def oracle(nums1, nums2, k):
    return sum(1 for x, y in product(nums1, nums2) if x % (y * k) == 0)


rng = random.Random(3162)
nonzero = 0
for trial in range(4000):
    n, m = rng.randint(1, 50), rng.randint(1, 50)
    pool = rng.choice([range(1, 51), [1, 2, 3, 4, 6, 12, 24, 48], [1, 2, 4, 8, 16, 32], range(1, 11)])
    nums1 = [rng.choice(list(range(1, 51)) if rng.random() < 0.5 else list(pool)) for _ in range(n)]
    nums2 = [rng.choice(list(pool)) for _ in range(m)]
    k = rng.choice([1, 1, 2, 3, rng.randint(1, 50)])
    assert 1 <= n <= 50 and 1 <= m <= 50 and 1 <= k <= 50
    assert all(1 <= v <= 50 for v in nums1 + nums2)
    ref = oracle(nums1, nums2, k)
    nonzero += ref > 0
    for key, fn in impls.items():
        assert fn(list(nums1), list(nums2), k) == ref, (key, nums1, nums2, k)
print("PASS 4000 bộ ngẫu nhiên, trong đó", nonzero, "bộ có đáp án khác 0")

count = 0
for nums1 in product([1, 2, 3, 4, 6], repeat=3):
    for nums2 in product([1, 2, 3], repeat=2):
        for k in (1, 2, 3):
            ref = oracle(nums1, nums2, k)
            for key, fn in impls.items():
                assert fn(list(nums1), list(nums2), k) == ref, (key, nums1, nums2, k)
            count += 1
print("PASS vét cạn", count, "bộ nhỏ")

PASS 4000 bộ ngẫu nhiên, trong đó 3377 bộ có đáp án khác 0
PASS vét cạn 3375 bộ nhỏ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Thiếu ngoặc, `x % y * k` thay vì `x % (y * k)`

In [26]:
def sai_thieu_ngoac(nums1, nums2, k):
    count = 0
    for x in nums1:
        for y in nums2:
            if x % y * k == 0:             # SAI: tính (x % y) * k
                count += 1
    return count


print("sai:", sai_thieu_ngoac([4], [2], 3), "| đúng:", Solution().numberOfPairs([4], [2], 3))
print("sai:", sai_thieu_ngoac([1, 2, 4, 12], [2, 4], 3), "| đúng:", Solution().numberOfPairs([1, 2, 4, 12], [2, 4], 3))

sai: 1 | đúng: 0
sai: 5 | đúng: 2


**Vì sao sai:** `%` và `*` cùng mức ưu tiên, tính từ trái sang phải: `4 % 2 * 3 = (4 % 2) * 3 = 0 * 3 = 0` nên bị coi là "chia hết", trong khi `4` **không** chia hết cho `2 * 3 = 6`. **Cách sửa:** `x % (y * k) == 0`.

### Lỗi 2: Quên nhân `k`

In [27]:
def sai_quen_k(nums1, nums2, k):
    return sum(1 for x in nums1 for y in nums2 if x % y == 0)


print("sai:", sai_quen_k([1, 2, 4, 12], [2, 4], 3), "| đúng:", Solution().numberOfPairs([1, 2, 4, 12], [2, 4], 3))
print("sai:", sai_quen_k([1, 3, 4], [1, 3, 4], 1), "| đúng:", Solution().numberOfPairs([1, 3, 4], [1, 3, 4], 1), "(trùng vì k = 1)")

sai: 5 | đúng: 2
sai: 5 | đúng: 5 (trùng vì k = 1)


**Vì sao sai:** ví dụ có `k = 1` che mất lỗi này (nhân `1` không đổi gì). Ở Ví dụ 2 `k = 3` thì đáp án bị lệch. **Bài học:** luôn thử ví dụ có `k > 1`. **Cách sửa:** dùng `y * k` làm số chia.

### Lỗi 3: Đảo vai trò hai mảng

In [28]:
def sai_dao_vai_tro(nums1, nums2, k):
    return sum(1 for x in nums1 for y in nums2 if y % (x * k) == 0)      # SAI: nums2[j] chia hết cho nums1[i] * k


print("sai:", sai_dao_vai_tro([1, 2, 4, 12], [2, 4], 3), "| đúng:", Solution().numberOfPairs([1, 2, 4, 12], [2, 4], 3))
print("sai:", sai_dao_vai_tro([1, 3, 4], [1, 3, 4], 1), "| đúng:", Solution().numberOfPairs([1, 3, 4], [1, 3, 4], 1), "(trùng vì hai mảng giống nhau)")

sai: 0 | đúng: 2
sai: 5 | đúng: 5 (trùng vì hai mảng giống nhau)


**Vì sao sai:** đề yêu cầu **`nums1[i]` chia hết cho `nums2[j] * k`** (số bị chia là `nums1`, số chia là `nums2 * k`). Đảo lại là đếm một bài toán khác. Ví dụ 1 có hai mảng **giống nhau** nên không phát hiện được lỗi — ví dụ 2 mới lộ. **Cách sửa:** `x` lấy từ `nums1` đứng bên **trái** `%`.

### Lỗi 4: Dùng `/` thay vì `%`

In [29]:
def sai_chia_thuc(nums1, nums2, k):
    count = 0
    for x in nums1:
        for y in nums2:
            if x / (y * k) == int(x / (y * k)):          # chạy được với số nhỏ, nhưng dễ sai số và vòng vo
                count += 1
    return count


print(sai_chia_thuc([1, 2, 4, 12], [2, 4], 3), "(đúng với số nhỏ, nhưng không nên dùng)")
print("0.1 + 0.2 == 0.3?", 0.1 + 0.2 == 0.3)           # lý do tránh so sánh số thực
print("cách tốt:", Solution().numberOfPairs([1, 2, 4, 12], [2, 4], 3))

2 (đúng với số nhỏ, nhưng không nên dùng)
0.1 + 0.2 == 0.3? False
cách tốt: 2


**Vì sao nên tránh:** `/` cho **số thực**; so sánh bằng với số thực có thể sai khi số lớn (sai số làm tròn). Với số nguyên, **`%` chính xác tuyệt đối**. Một biến thể khác cũng nên nhớ: `//` là chia nguyên, còn kiểm tra chia hết chuẩn là `x % d == 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Đếm số cặp `(i, j)` thỏa điều kiện**" với `n, m` nhỏ (`<= 50`, `<= 1000`) ⇒ **hai vòng lặp** là đủ; đừng tối ưu sớm.
- "**`a` chia hết cho `b`**" ⇒ `a % b == 0`; nếu có tích `b * k` ⇒ **luôn có ngoặc**.
- Nhiều giá trị trùng ⇒ **`Counter`**, nhân số lần của hai phía.
- Khi dữ liệu lớn (bài II: `n, m` tới `10^5`, giá trị tới `10^6`) ⇒ **liệt kê bội số** (chuỗi điều hòa) thay vì xét mọi cặp.
- `x >= d` là điều kiện cần để `d` chia hết `x` (khi `x > 0`) ⇒ cắt tỉa nhanh.
- Kiểm tra ví dụ có `k > 1` và hai mảng **khác nhau** để lộ lỗi quên `k` / đảo vai trò.

## 15. Biến thể của bài

1. **Find the Number of Good Pairs II (3164):** cùng đề nhưng `n, m <= 10^5`, giá trị `<= 10^6` — dùng `Counter` + liệt kê bội số (Cách 5).
2. **Liệt kê các cặp tốt** (không chỉ đếm).
3. **Cặp tốt theo điều kiện khác:** `nums1[i] % nums2[j] == k`, hoặc `nums1[i] + nums2[j]` chia hết cho `k`.
4. **Một mảng:** đếm cặp `(i, j)` với `i < j` mà `nums[j]` chia hết cho `nums[i]`.
5. Bài gần gũi: **Count Pairs Whose Sum is Less than Target (2824)**, **Number of Pairs of Strings With Concatenation Equal to Target (2023)**, **Number of Good Pairs (1512)**.

In [30]:
def list_good_pairs(nums1, nums2, k):
    return [(i, j) for i, x in enumerate(nums1) for j, y in enumerate(nums2) if x % (y * k) == 0]


def count_divisible_in_one_array(nums):
    """Số cặp (i, j), i < j, mà nums[j] chia hết cho nums[i]."""
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) if nums[j] % nums[i] == 0)


def count_pairs_remainder(nums1, nums2, k):
    """Số cặp (i, j) mà nums1[i] % nums2[j] == k."""
    return sum(1 for x in nums1 for y in nums2 if x % y == k)


print(list_good_pairs([1, 3, 4], [1, 3, 4], 1))             # [(0, 0), (1, 0), (1, 1), (2, 0), (2, 2)]
print(count_divisible_in_one_array([1, 2, 4, 3]))            # (1,2), (1,4), (1,3), (2,4) -> 4
print(count_pairs_remainder([5, 7, 9], [2, 4], 1))           # 5%2=1, 5%4=1, 7%2=1, 7%4=3, 9%2=1, 9%4=1 -> 5
assert list_good_pairs([1, 3, 4], [1, 3, 4], 1) == [(0, 0), (1, 0), (1, 1), (2, 0), (2, 2)]
assert list_good_pairs([1, 2, 4, 12], [2, 4], 3) == [(3, 0), (3, 1)]
assert count_divisible_in_one_array([1, 2, 4, 3]) == 4
assert count_pairs_remainder([5, 7, 9], [2, 4], 1) == 5

[(0, 0), (1, 0), (1, 1), (2, 0), (2, 2)]
4
5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng `for` | xét mọi `(i, j)`, đếm khi chia hết | `O(n * m)` | `O(1)` | **nên nộp**: rõ ràng nhất với `n, m <= 50` |
| 2. `sum` + generator | cùng ý, viết một dòng | `O(n * m)` | `O(1)` | thích ngắn gọn |
| 3. Tính trước `d`, cắt tỉa | `d = y * k` tính một lần, bỏ `d > x` | `O(n * m)` | `O(m)` | giảm hằng số |
| 4. `Counter` hai bên | kiểm tra theo giá trị, nhân số lần | `O(u1 * u2)` | `O(u1 + u2)` | nhiều giá trị trùng |
| 5. Liệt kê bội số | mỗi `y`: các bội số của `y * k` | `~O(V log V)` | `O(u1 + u2)` | dữ liệu lớn (bài II) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `18 % 6`, `18 % 4`, `10 % 2 * 3`, `10 % (2 * 3)`, `sum(x > 2 for x in [1, 2, 3, 4])`.

**Bài 2 (tính tay).** Với `nums1 = [2, 6, 9, 18]`, `nums2 = [1, 3]`, `k = 2`: tính `d = y * k`, lập bảng kiểm tra từng `x`, rồi cho biết số cặp tốt.

**Bài 3 (điền TODO).** Hoàn thành `number_of_pairs_todo(nums1, nums2, k)` theo Cách 4 (`Counter`).

In [31]:
def number_of_pairs_todo(nums1, nums2, k):
    # TODO 1: c1 = Counter(nums1), c2 = Counter(nums2), total = 0
    # TODO 2: với mỗi (x, a) trong c1.items() và (y, b) trong c2.items():
    #         nếu x % (y * k) == 0 thì total += a * b
    # TODO 3: trả về total
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `number_of_pairs_zip_free(nums1, nums2, k)` chỉ dùng **một** vòng `for` ngoài trên `nums1` và với mỗi `x` đếm số `y` thỏa điều kiện bằng `sum(...)` (không có vòng `for` lồng dạng câu lệnh).

**Bài 5 (biến thể / mini challenge).** (a) Viết `number_of_pairs_ii(nums1, nums2, k)` hiệu quả theo kiểu **liệt kê bội số** để dùng cho dữ liệu lớn hơn (giá trị tới `10^6`): tính `max(nums1)` và dùng `Counter`. (b) Viết `good_pairs_by_row(nums1, nums2, k)` trả về danh sách `row[i]` = số `j` sao cho `(i, j)` là cặp tốt.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `18 % 6 = 0`; `18 % 4 = 2`; `10 % 2 * 3 = (10 % 2) * 3 = 0`; `10 % (2 * 3) = 10 % 6 = 4`; `sum(x > 2 ...) = 2`. Bài 3: nhớ nhân `a * b`.
- **Mức 2 (vừa):** Bài 2: `d = 2` và `d = 6`. `x = 2`: chia hết cho 2 (có), cho 6 (không) ⇒ 1. `x = 6`: có, có ⇒ 2. `x = 9`: không, không ⇒ 0. `x = 18`: có, có ⇒ 2. Tổng `5`. Bài 4: `for x in nums1: count += sum(x % (y * k) == 0 for y in nums2)`.
- **Mức 3 (sâu):** Bài 5(a): với mỗi `y` khác nhau, `d = y * k`, `for m in range(d, limit + 1, d): total += c1[m] * b`. 5(b): `[sum(x % (y * k) == 0 for y in nums2) for x in nums1]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert 18 % 6 == 0
assert 18 % 4 == 2
assert 10 % 2 * 3 == 0
assert 10 % (2 * 3) == 4
assert sum(x > 2 for x in [1, 2, 3, 4]) == 2

# Bài 2
assert Solution().numberOfPairs([2, 6, 9, 18], [1, 3], 2) == 1 + 2 + 0 + 2 == 5


# Bài 3
def number_of_pairs_todo_ans(nums1, nums2, k):
    c1, c2 = Counter(nums1), Counter(nums2)
    total = 0
    for x, a in c1.items():
        for y, b in c2.items():
            if x % (y * k) == 0:
                total += a * b
    return total


for nums1, nums2, k, expected in examples:
    assert number_of_pairs_todo_ans(nums1, nums2, k) == expected


# Bài 4
def number_of_pairs_zip_free(nums1, nums2, k):
    count = 0
    for x in nums1:
        count += sum(x % (y * k) == 0 for y in nums2)
    return count


for nums1, nums2, k, expected in examples:
    assert number_of_pairs_zip_free(nums1, nums2, k) == expected


# Bài 5(a): dùng được cả cho giá trị lớn (tới 10^6) nhờ liệt kê bội số
def number_of_pairs_ii(nums1, nums2, k):
    c1, c2 = Counter(nums1), Counter(nums2)
    limit = max(nums1)
    total = 0
    for y, b in c2.items():
        d = y * k
        for m in range(d, limit + 1, d):
            total += c1[m] * b
    return total


for nums1, nums2, k, expected in examples:
    assert number_of_pairs_ii(nums1, nums2, k) == expected
big1 = [10 ** 6, 999999, 500000, 10 ** 6]
big2 = [1, 2, 500000, 10 ** 6]
assert number_of_pairs_ii(big1, big2, 1) == oracle(big1, big2, 1)
assert number_of_pairs_ii(big1, big2, 3) == oracle(big1, big2, 3)


# Bài 5(b)
def good_pairs_by_row(nums1, nums2, k):
    return [sum(x % (y * k) == 0 for y in nums2) for x in nums1]


assert good_pairs_by_row([1, 3, 4], [1, 3, 4], 1) == [1, 2, 2]
assert good_pairs_by_row([1, 2, 4, 12], [2, 4], 3) == [0, 0, 0, 2]
assert sum(good_pairs_by_row([1, 3, 4], [1, 3, 4], 1)) == 5
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm cặp `(i, j)` mà `nums1[i]` chia hết cho `nums2[j] * k`)

```python
class Solution:
    def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int:
        count = 0
        for x in nums1:
            for y in nums2:
                if x % (y * k) == 0:
                    count += 1
        return count
```

**Checklist**

- [ ] Có **ngoặc** `x % (y * k)` (không viết `x % y * k`)?
- [ ] `x` lấy từ `nums1` (số bị chia), `y` lấy từ `nums2` (số chia), và đã nhân `k`?
- [ ] Đếm theo **chỉ số** (giá trị trùng vẫn tính nhiều cặp)?
- [ ] Dùng `%` trên số nguyên, không dùng `/`?
- [ ] Đã thử cả hai ví dụ (`5` và `2`), trường hợp `k > 1`, và trường hợp giá trị trùng (`[2, 2]` với `[1]`)?
- [ ] Skeleton đúng `class Solution` / `def numberOfPairs(self, nums1: List[int], nums2: List[int], k: int) -> int`?